In [2]:
using QEDprocesses
using QEDcore
using QEDbase

In [8]:
N = 2
FLOAT_T = Float64

OMEGA = FLOAT_T(1000)
MODEL = PerturbativeQED()
PROCESS = ScatteringProcess(
    (Electron(), Photon()),                                                         # incoming particles
    (Electron(), ntuple(_ -> Electron(), N)..., ntuple(_ -> Positron(), N)...),     # outgoing particles
    (SyncedSpin(1), AllPol()),
    (SyncedSpin(1), ntuple(_ -> SyncedSpin(1), N)..., ntuple(_ -> SyncedSpin(2), N)...),
)
IN_PSL = TwoBodyTargetSystem()
PSL = FlatPhaseSpaceLayout(IN_PSL)

FlatPhaseSpaceLayout{TwoBodyTargetSystem{Energy{2}}}(TwoBodyTargetSystem{Energy{2}}(Energy{2}()))

In [9]:
psp = PhaseSpacePoint(
    PROCESS,
    MODEL,
    PSL,
    (OMEGA,),
    tuple((rand(FLOAT_T) for _ in 1:phase_space_dimension(PROCESS, MODEL, PSL))...)
)

PhaseSpacePoint:
    process: generic QED process "ek -> eeepp"
    model: perturbative QED
    phase space layout: FlatPhaseSpaceLayout{TwoBodyTargetSystem{Energy{2}}}(TwoBodyTargetSystem{Energy{2}}(Energy{2}()))
    incoming particles:
     -> incoming electron: [1.0, 0.0, 0.0, 0.0]
     -> incoming photon: [1000.0, 0.0, 0.0, 1000.0]
    outgoing particles:
     -> outgoing electron: [11.052187300098561, -0.8473085952725702, -0.5093432581112781, 10.962366610648159]
     -> outgoing electron: [110.44269114447623, 2.4163835123272026, -0.04924583667514898, 110.41171447270932]
     -> outgoing electron: [270.3113330723366, -7.277618485444495, -5.356853284154885, 270.1583927613874]
     -> outgoing positron: [23.19735832466687, -4.897430040996799, 1.777995696820276, 22.582545107640527]
     -> outgoing positron: [585.9964301584429, 10.605973609386663, 4.137446682121036, 585.8849810476356]


In [10]:
QEDbase.unsafe_differential_cross_section(psp)

┌ Info: built graph
└ @ QEDprocesses /home/antonr/repos/QEDprocesses.jl/src/processes/generic_process/perturbative/cross_section.jl:30
┌ Info: Graph:
│   Nodes: Total: 4716, QEDFeynmanDiagrams.ComputeTask_BaseState: 14, ComputableDAGs.DataTask: 2395, 
│          QEDFeynmanDiagrams.ComputeTask_CollectTriples: 8, QEDFeynmanDiagrams.ComputeTask_TripleNegated: 1200, QEDFeynmanDiagrams.ComputeTask_CollectPairs: 246, 
│          QEDFeynmanDiagrams.ComputeTask_PropagatePairs: 246, QEDFeynmanDiagrams.ComputeTask_SpinPolCumulation: 1, QEDFeynmanDiagrams.ComputeTask_Pair: 256, 
│          QEDFeynmanDiagrams.ComputeTask_PairNegated: 50, QEDFeynmanDiagrams.ComputeTask_Triple: 240, QEDFeynmanDiagrams.ComputeTask_Propagator: 60
│   Edges: 9573
│   Total Compute Effort: 0.0
│   Total Data Transfer: 0.0
│   Total Compute Intensity: 0.0
└ @ QEDprocesses /home/antonr/repos/QEDprocesses.jl/src/processes/generic_process/perturbative/cross_section.jl:31
┌ Info: continuing
└ @ QEDprocesses /home/antonr/repo

1.7055748641887918e-14

In [11]:
using ProgressMeter

N_EVENTS = 100_000
EVENTS = Matrix{FLOAT_T}(undef, (N_EVENTS, (phase_space_dimension(PROCESS, MODEL, PSL) + 1)))

@showprogress for i in 1:N_EVENTS
    coords = tuple((rand(FLOAT_T) for _ in 1:phase_space_dimension(PROCESS, MODEL, PSL))...)
    psp = PhaseSpacePoint(
        PROCESS,
        MODEL,
        PSL,
        (OMEGA,),
        coords,
    )

    diff_cs = unsafe_differential_cross_section(psp)
    EVENTS[i, :] = [diff_cs, coords...]
end

Progress: 100%|█████████████████████████████████████████| Time: 0:00:16


In [12]:
using JLD2

@save "$(N)_pair_shower_events.jld2" EVENTS